# Assignment 34: Text Summarization using LangChain

**Student:** Abhishek Thakare

Four different ways to summarize the same long article - a plain prompt,
LangChain's "stuff" chain, "map-reduce", and "refine" - built one at a time so
I can actually compare them on the same input instead of just reading about
the differences.

I wrote an original ~1,300-word article for this (`data/long_article.txt`,
about 7,850 characters) on Retrieval-Augmented Generation, mainly because it
ties into everything I've already built in earlier assignments, and because
writing my own text means there's no copyright question about summarizing
it.

**A version note that's actually relevant here, not just my usual OpenAI
situation:** `load_summarize_chain` - the function Tasks 5, 8, and 11
specifically ask for - lives in LangChain's older "legacy chains" API and was
removed when LangChain hit version 1.0. I pinned `langchain==0.3.27` and
`langchain-community==0.3.27` in `requirements.txt` specifically so this
exact function is available, since that's the API this assignment is built
around. I checked this by actually trying to import it under the newest
LangChain first, watching it fail, then confirming it works once pinned to
0.3.27 - it's a real compatibility issue, not something I'm guessing at.

**Same honest note as my other recent assignments:** no working OpenAI
credits, so this is **Ollama running `llama3.2`** locally, since the
restriction is "LangChain + LLM of choice."


## Before running this

- Ollama running locally with `llama3.2` pulled.
- **Important:** `pip install -r requirements.txt` installs the pinned
  `langchain==0.3.27` / `langchain-community==0.3.27` - installing a plain
  `pip install langchain` instead will get you the newest 1.x version, where
  `load_summarize_chain` doesn't exist anymore and Tasks 5/8/11 will fail on
  import.
- `data/long_article.txt` in a `data/` folder next to this notebook.


In [1]:
# Run this only if something is missing in your environment
# %pip install -U "langchain==0.3.27" "langchain-community==0.3.27" langchain-text-splitters langchain-ollama

In [2]:
from summarizer import load_text

article = load_text("data/long_article.txt")
print("Loaded article length:", len(article), "characters")


c:\Users\abhis\Desktop\Tutedude_Course\GenAI-Task34-AbhishekThakare\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Loaded article length: 7853 characters


## PART 1 — Basic Text Summarisation using PromptTemplate

### Task 1: Load and Prepare Text

Just a plain file read here - the assignment allows manual loading as an
alternative to a LangChain document loader for this part, and a `.txt` file
doesn't really need anything fancier.


In [3]:
print("Total characters:", len(article))
print("\nSample content preview:\n")
print(article[:400])


Total characters: 7853

Sample content preview:

The Rise of Retrieval-Augmented Generation in Enterprise AI

Large language models changed what people expect from software almost overnight. Ask a
model a question and it responds in fluent, confident prose - no menus, no search results
page, just an answer. But that fluency hides a real limitation: a model only knows what it
saw during training, frozen at some cutoff date, and it has no built-in


### Task 2: Prompt-Based Summarization

A plain `PromptTemplate` - no chain abstraction at all yet - with a system-
style instruction telling the model it's acting as a summarizer, and a
placeholder for the actual text.


In [4]:
from summarizer import get_llm, short_summary_prompt, summarize_with_prompt

llm = None
try:
    llm = get_llm()
    llm.invoke("say ok")
    print("Ollama is up, llama3.2 responded.")
except Exception as e:
    llm = None
    print("Couldn't reach Ollama:", e)
    print("The summarization cells below will print a placeholder instead of a real summary.")


Ollama is up, llama3.2 responded.


In [5]:
print(short_summary_prompt.template)


You are an expert summarizer. Summarize the following text in 5-6 lines, capturing only the most important points.

Text:
{text}

Summary:


In [6]:
try:
    summary = summarize_with_prompt(article, style="short", llm=llm)
    print(summary)
except Exception as e:
    print("Summarization failed:", e)


Here is a 5-6 line summary of the text:

Retrieval-Augmented Generation (RAG) is a solution to the limitation of large language models, which can only recall information seen during training. RAG involves retrieving relevant documents from a knowledge base and handing them to the model as extra context. This approach reduces "hallucination" and improves accuracy. RAG systems use vector databases and embedding models to efficiently search and retrieve relevant information. The technology is modular, allowing for easy upgrades and improvements. RAG has become the default architecture for enterprise document Q&A due to its effectiveness and scalability.


### Task 3: Prompt Variations

Two versions of the same idea - a tight 5-6 line summary, and a bulleted
list of key points - run on the exact same article so the *style* difference
is the only variable.


In [7]:
from summarizer import summarize_with_prompt

try:
    short_version = summarize_with_prompt(article, style="short", llm=llm)
    bullet_version = summarize_with_prompt(article, style="bullet", llm=llm)

    print("--- Short summary (5-6 lines) ---")
    print(short_version)
    print("\n--- Bullet-point summary ---")
    print(bullet_version)
except Exception as e:
    print("Failed:", e)


--- Short summary (5-6 lines) ---
Here is a summary of the text in 6 lines, capturing the most important points:

Retrieval-Augmented Generation (RAG) is a solution to the limitation of large language models, which can only recall information seen during training. RAG involves retrieving relevant documents from a knowledge base and handing them to the model as extra context. This approach reduces the "lost in the middle" effect and improves the model's ability to answer questions. RAG systems use vector databases and embedding models to efficiently search and retrieve relevant chunks of text. To improve performance, RAG systems often add refinements such as maximal marginal relevance and multi-query retrieval. RAG has become the default architecture for enterprise document Q&A due to its modularity and effectiveness.

--- Bullet-point summary ---
Here is a concise bulleted list of the key points about Retrieval-Augmented Generation (RAG) in Enterprise AI:

• RAG is a solution to the li

Couldn't actually compare real outputs here since Ollama wasn't reachable in
this environment - but the setup is right: same article, same underlying
instruction to summarize, only the requested *format* changes between the two
prompts. If this were working, I'd expect the short version to read like a
paragraph and the bullet version to break the same points into a scannable
list rather than prose.


## PART 2 — Stuff Summarization Chain

### Task 4: Why Stuff Chain is Needed (Conceptual)

**What is a stuff chain?** It's the simplest possible summarization chain -
"stuff" all the document chunks into a single prompt, in one shot, and ask
the LLM to summarize the whole thing at once. No splitting logic beyond
however the documents were already chunked, no multiple LLM calls - just one
big prompt in, one summary out.

**When is it suitable?** When the whole document (or all its chunks combined)
comfortably fits inside the model's context window in a single prompt. For a
short article, a few pages, or a handful of short documents, stuffing
everything into one prompt is the most direct approach and doesn't add the
overhead of multiple LLM calls.

**Limitations:** It falls over completely once the combined text is larger
than the model's context window - there's no fallback, it just won't fit.
Even when it technically fits, a very long single prompt can suffer from
models paying less attention to information buried in the middle of a long
block of text, so quality can degrade well before the hard size limit is
even reached.


### Task 5: Implement Stuff Summarization Chain

Splitting the article into `Document` chunks first (needed for
`load_summarize_chain` regardless of chain type), then running the stuff
chain over the whole set at once.


In [8]:
from summarizer import build_documents, summarize_stuff

article_docs = build_documents(article)
print("Number of chunks:", len(article_docs))
for i, d in enumerate(article_docs):
    print(f"  chunk {i}: {len(d.page_content)} chars")


Number of chunks: 9
  chunk 0: 794 chars
  chunk 1: 671 chars
  chunk 2: 1034 chars
  chunk 3: 762 chars
  chunk 4: 493 chars
  chunk 5: 1188 chars
  chunk 6: 973 chars
  chunk 7: 1069 chars
  chunk 8: 1032 chars


In [9]:
try:
    stuff_summary = summarize_stuff(article_docs, llm=llm)
    print(stuff_summary)
except Exception as e:
    print("Stuff chain failed:", e)


Retrieval-Augmented Generation (RAG) is a technique used to improve the performance of large language models in answering questions about internal company knowledge. The main limitation of large language models is that they only know what they were trained on and cannot recall information outside of that training data.

RAG addresses this limitation by first retrieving relevant documents from a knowledge base and then using the retrieved context to answer the question. This approach, known as the "retrieval step," allows the model to access information that it was not trained on.

The technique involves several key components:

1. Retrieval: Converting documents into small chunks and representing each chunk as a vector, which is then stored in a database.
2. Grounding: Instructing the model to answer only from the provided block of retrieved context, and telling it to say it doesn't know if the answer isn't present in that context.
3. Smarter retrieval: Refining the retrieval step by u

### Task 6: Comparison with Prompt-Based Summary

Couldn't run either one for real in this environment, but structurally:
the Task 2 prompt-based version manually formats one string and sends it
straight to the LLM - it's really just a `PromptTemplate | LLM` pipeline with
no awareness of "documents" as a concept at all. The stuff chain does
effectively the same thing (one call, everything at once) but through
LangChain's `load_summarize_chain` abstraction, which expects a list of
`Document` objects and handles formatting them into the prompt internally.
For an article this size, I'd expect the two to produce very similar
summaries, since they're doing the same fundamental thing - the real value
of the chain abstraction shows up once I swap in `map_reduce` or `refine`
without changing how I call it.


## PART 3 — Map-Reduce Summarization Chain

### Task 7: Why Map-Reduce is Needed (Conceptual)

**Why do large documents need map-reduce?** Once a document (or set of
documents) is too large to fit in one prompt at all, "stuff everything into
one call" simply isn't an option anymore - there's nothing to stuff it into.

**How map and reduce work:** The "map" step summarizes each chunk
*independently*, one LLM call per chunk, producing a short summary for every
piece of the original document. The "reduce" step then takes all of those
per-chunk summaries and combines them into one final summary - effectively
summarizing the summaries. This means the model never has to see the whole
original document at once; it only ever sees one chunk (during map) or a
handful of already-short summaries (during reduce), so the approach scales
to documents of essentially any length by just adding more chunks and more
map calls.


### Task 8: Implement Map-Reduce Summarization Chain

Same chunked `article_docs` from Task 5, just run through
`chain_type="map_reduce"` instead of `"stuff"`.


In [10]:
from summarizer import summarize_map_reduce

try:
    map_reduce_summary = summarize_map_reduce(article_docs, llm=llm)
    print(map_reduce_summary)
except Exception as e:
    print("Map-reduce chain failed:", e)


c:\Users\abhis\Desktop\Tutedude_Course\GenAI-Task34-AbhishekThakare\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abhis\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Here is a concise summary:

RAG (Retrieval-Augmented Generation) is a method that combines language model generation with document retrieval to provide more accurate answers. It uses a retrieval step to break down documents into small chunks and search for the closest matching chunks to answer questions. This approach reduces hallucinations and improves accuracy. RAG also employs smarter retrieval techniques, such as maximal marginal relevance and contextual compression, to improve efficiency. The system's modular architecture allows for easy upgrades and modifications, making it suitable for enterprise document Q&A.


### Task 9: Analyze Map Outputs (Optional)

Turning on `return_intermediate_steps` to actually see each chunk's
individual summary before the reduce step combines them - this is the part
that makes map-reduce feel less like a black box.


In [11]:
try:
    final_summary, intermediate_summaries = summarize_map_reduce(
        article_docs, llm=llm, return_intermediate_steps=True
    )

    print("--- Intermediate (per-chunk) summaries ---")
    for i, step in enumerate(intermediate_summaries):
        print(f"\nChunk {i} summary:")
        print(step)

    print("\n--- Final combined summary ---")
    print(final_summary)
except Exception as e:
    print("Failed:", e)
    print("(Would show one summary per chunk here, then how the reduce step merges them.)")


--- Intermediate (per-chunk) summaries ---

Chunk 0 summary:
Large language models have limitations in their ability to recall information not seen during training, making them unsuitable for tasks requiring knowledge of internal company data or policies.

Chunk 1 summary:
Retrieval-Augmented Generation (RAG) is a technique that combines language model generation with document retrieval to provide more accurate and context-specific answers. It involves searching for relevant documents and providing them to the model as extra context, allowing it to generate answers based on the actual information rather than relying on memorized knowledge.

Chunk 2 summary:
Naive prompting, where a large block of text is pasted into a single prompt, is not an effective solution for complex questions. This approach breaks down quickly as the amount of text grows, and models struggle to extract relevant information from the middle of long prompts, leading to inefficiencies and reduced accuracy.

Chunk 3 

If this were running for real, I'd expect each intermediate summary to only
reflect its own chunk (so the chunk about "grounding and hallucination"
wouldn't mention map-reduce at all, for instance, since that's a different
section entirely), and the final reduce step to read like a summary *of
summaries* - broader and a bit more compressed than any single intermediate
one, since it has to fold several already-condensed points into one.


## PART 4 — Refine Summarization Chain

### Task 10: Understanding Refine Chain (Conceptual)

**How refine works:** Refine processes chunks *sequentially* rather than in
parallel. It summarizes the first chunk, then for every following chunk it
shows the model the summary-so-far plus the new chunk and asks it to refine
(update) that summary to incorporate the new information - one running
summary that gets revised chunk by chunk, rather than a big batch of
independent summaries combined at the end.

**Difference between map-reduce and refine:** Map-reduce summarizes every
chunk independently and in isolation - chunk 5's summary has no idea what
chunk 2 said - then merges everything in one final reduce step. Refine
instead carries a single evolving summary through every chunk in order, so
each step's output already reflects everything that came before it. That
sequential dependency means refine can't run its steps in parallel the way
map-reduce's map step can, but it can produce a more coherent, better-flowing
summary since each step is explicitly building on the last rather than being
combined after the fact.


### Task 11: Implement Refine Summarization Chain


In [12]:
from summarizer import summarize_refine

try:
    refine_summary = summarize_refine(article_docs, llm=llm)
    print(refine_summary)
except Exception as e:
    print("Refine chain failed:", e)


Here is a refined summary:

Large language models have limitations, particularly in recalling information not seen during training. To address this, Retrieval-Augmented Generation (RAG) is being explored as a potential solution. RAG involves retrieving relevant documents from external sources and providing them to the language model as extra context alongside a question, allowing the model to answer using the provided information rather than relying on memorized knowledge.

This approach, known as grounding, helps to reduce "hallucination" - the production of confident but incorrect answers. By explicitly instructing the model to answer only from the provided context and telling it to say "I don't know" when the answer isn't present, hallucination drops substantially. This is achieved by using a system prompt that clearly communicates the boundaries of the context.

RAG systems can efficiently search for relevant chunks of text that match a question's vector, providing a targeted slice

### Task 12: Comparison of All Summarization Methods

| Method | Summary quality | Coherence | Suitability for long docs |
|---|---|---|---|
| Prompt-based | Good for short text; degrades once the text approaches the context limit | High - single pass, one voice throughout | Poor - no chunking at all |
| Stuff chain | Same as prompt-based in practice, just via the chain abstraction | High - also a single pass | Poor - still one prompt, just built differently |
| Map-Reduce | Good coverage of every chunk, but the reduce step can feel like a list of points stitched together | Lower - each chunk summarized in isolation, reduce step has to reconcile pieces that never saw each other | Excellent - scales to any length, map calls can run in parallel |
| Refine | Tends to read most naturally since each step explicitly builds on the last | Highest of the chunked approaches - one continuously-updated summary | Good - handles long documents, but sequential calls mean it's slower than map-reduce |

Couldn't fill this table in with real observed differences since none of the
chains actually ran end to end in this environment - it's built from how each
method is structurally supposed to behave rather than from comparing genuine
output side by side.


## PART 5 — Mini Project: Document Summarizer

### Task 13: Build a Summarization Function

Pulled everything into `summarize_document(text, method="map_reduce")` in
`summarizer.py`, switching between all four approaches from one place.
Testing that it actually routes to the right method, and that it rejects an
unrecognized one instead of silently doing something odd.


In [14]:
from summarizer import summarize_document

for method in ["prompt", "stuff", "map_reduce", "refine"]:
    print(f"--- {method} ---")
    try:
        print(summarize_document(article, method=method, llm=llm))
    except Exception as e:
        print(f"[failed - {e}]")
    print()

try:
    summarize_document(article, method="bogus", llm=llm)
except ValueError as e:
    print("Correctly rejected an unknown method:", e)


--- prompt ---
Here is a 6-line summary of the text:

Retrieval-Augmented Generation (RAG) is a solution to the limitation of large language models, which only know what they were trained on. RAG involves retrieving relevant documents from a knowledge base and using them as context to answer questions. This approach reduces hallucination and improves accuracy. RAG systems use vector databases to store and retrieve chunks of text, allowing for efficient and scalable querying. To address follow-up questions, RAG systems keep a running conversation history and use it to inform the prompt. RAG has become the default architecture for enterprise document Q&A due to its modularity and ability to adapt to improving underlying models.

--- stuff ---
Retrieval-Augmented Generation (RAG) is a technique used to improve the performance of large language models in answering questions about internal company knowledge. The main limitation of large language models is that they only know what they were 

## Task 14: Observations & Insights

**1. Best summarization strategy for very long documents**
Map-reduce is the most practical default for genuinely long documents,
mainly because its per-chunk map calls are independent of each other and can
run in parallel, so it scales in wall-clock time as well as in raw document
length. Refine can produce a more coherent read since it's explicitly
building one continuous summary, but it has to run its steps in sequence
(each one depends on the previous summary), so it gets slower - not just
more expensive - as the document grows, in a way map-reduce doesn't.

**2. Trade-offs between speed and quality**
Prompt-based and stuff are the fastest since they're a single LLM call, but
that's only viable up to whatever fits in one context window. Map-reduce
trades some coherence (chunks summarized blind to each other) for both scale
and parallelizable speed. Refine trades speed (strictly sequential, one call
per chunk, no parallelism) for a more coherent final summary. There's no
single "best" option here - it's really "pick based on whether the document
fits in one prompt, and if not, whether raw speed or read-quality matters
more for this particular use case."

**3. Real-world use cases of each method**
*Prompt-based / stuff* fit short reports, single meeting transcripts, or
individual support tickets - anything that comfortably fits in one prompt.
*Map-reduce* fits large batches of documents that need summarizing fast, like
overnight summarization of a day's worth of customer support tickets, where
speed and independent processing matter more than a perfectly polished
narrative. *Refine* fits long-form single documents where readability really
matters, like summarizing a lengthy legal contract or a research paper into
one coherent, well-flowing paragraph rather than a set of independently
generated points stitched together.


## Final note

The most useful thing I actually ran into building this wasn't about
summarization strategy at all - it was discovering that `load_summarize_chain`
doesn't exist in current LangChain anymore. It's a good reminder that a
library moving fast enough to add things like LCEL and the newer `Runnable`
interface also means some older APIs get retired along the way, and pinning
versions deliberately (with a note explaining why) is sometimes the right
call rather than always reaching for the latest release.
